**W상태 시뮬레이션**

In [7]:
from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.visualization import plot_histogram

# ⚠ 여기 두 값은 네가 직접 넣어야 함. 이 파일은 절대 깃허브/친구한테 그대로 주지 마라.
API_KEY = "P02Z-dFbZIPqzuwKpdmvNbqor2Jtt44IMK-U96wOLFPs"
INSTANCE_CRN = "crn:v1:bluemix:public:quantum-computing:us-east:a/aa1d225111e942c1b7835898cdcb9ea6:ec09793f-467a-4640-bb56-44f65e6db61e::"

# 사용할 백엔드 고르기: 셋 중 아무거나
BACKEND_NAME = "ibm_torino" # ibm_fez, ibm_marrakesh 로 바꿔도 됨

service = QiskitRuntimeService(
    channel="ibm_quantum_platform",
    token=API_KEY,
    instance=INSTANCE_CRN,
)

backend = service.backend(BACKEND_NAME)
print("사용 백엔드:", backend.name, "| qubits:", backend.num_qubits)

# ==== 회로 생성 ====
qc = QuantumCircuit(3, 3)

amp = 1/(3**(1/2))
w_state = [
    0,      # |000>
    amp,    # |001>
    amp,    # |010>
    0,      # |011>
    amp,    # |100>
    0,      # |101>
    0,      # |110>
    0       # |111>
]

# q0, q1, q2 전체에 W 상태 초기화
qc.initialize(w_state, [0, 1, 2]) # 얘가 알아서 위 w_state 상황 만듬

# 나중에 확인하려고 측정 추가
qc.measure([0, 1, 2], [0, 1, 2])

print("원래 회로 : ")
print(qc)

# ==== 회로를 백엔드에 맞게 변환 ====
qc_t = transpile(qc, backend=backend)
print("Transpile된 회로:")
print(qc_t)

# ==== Sampler 실행 ====
sampler = Sampler(backend)
job = sampler.run([qc_t], shots=1500)
print("job id:", job.job_id())

result = job.result()
counts = result[0].data.c.get_counts()
print("측정 결과:", counts)

qiskit_runtime_service._discover_account:WARNING:2025-12-01 08:16:08,326: Loading account with the given token. A saved account will not be used.


사용 백엔드: ibm_torino | qubits: 133
원래 회로 : 
     ┌────────────────────────────────────────────────┐┌─┐      
q_0: ┤0                                               ├┤M├──────
     │                                                │└╥┘┌─┐   
q_1: ┤1 Initialize(0,0.57735,0.57735,0,0.57735,0,0,0) ├─╫─┤M├───
     │                                                │ ║ └╥┘┌─┐
q_2: ┤2                                               ├─╫──╫─┤M├
     └────────────────────────────────────────────────┘ ║  ║ └╥┘
c: 3/═══════════════════════════════════════════════════╩══╩══╩═
                                                        0  1  2 
Transpile된 회로:
global phase: π/4
               ┌─────────┐    ┌────┐   ┌──────────┐┌────┐┌────────────┐   »
q_1 -> 80 ─|0>─┤ Rz(π/2) ├────┤ √X ├───┤ Rz(-π/4) ├┤ √X ├┤ Rz(1.6709) ├─■─»
               └──┬────┬─┘┌───┴────┴──┐└──┬────┬──┘└────┘└────────────┘ │ »
q_2 -> 81 ─|0>────┤ √X ├──┤ Rz(1.231) ├───┤ √X ├────────────────────────■─»
               ┌──┴────┴─┐└───┬────┬